# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> dataset with the [`mlcroissant`](https://github.com/mlcommons/croissant) library using [Croissant schema](https://croissant.mlcommons.org/). All references to dataset structure entities (record sets, fields, columns) are made via their Croissant `@id` fields per best practice.

### Dataset Source

Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load dataset and metadata from the Croissant schema. The `mlcroissant` library handles fetching the schema, resolving distribution files, and gives access to machine-actionable metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.simplefilter('ignore')

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nIdentifier: {metadata.identifier}")
print(f"Version: {metadata.version if hasattr(metadata, 'version') else 'N/A'}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")

## 2. Data Overview

In Croissant, data is organized into one or more *record sets*, each with unique `@id`s. Within each record set, the data schema is defined by fields (also referenced by `@id`).

Let's enumerate the available record sets and their fields by `@id`.

In [ ]:
# List all record sets by their @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):")
for rs in record_sets:
    print(f"- Record set id: {rs['@id']} ; name: {rs.get('name', '(no name)')}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print(f"  Fields (@id):")
        for field in fields:
            fid = field['@id'] if isinstance(field, dict) else field
            print(f"    - {fid}")
    else:
        print("  No fields defined.")
    print()

## 3. Data Extraction

Below, we load data from each record set found above, using their `@id` for direct referencing. Each table is stored as a DataFrame in a dictionary keyed by the record set `@id`.

**Tip:** Replace `<record_set_id>` below with the desired record set's `@id`.

In [ ]:
# List record set @id(s)
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # Load all records for this record set
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded record set '{record_set_id}' with {len(df)} rows and {len(df.columns)} columns.")
    print(f"Columns: {list(df.columns)}\n")

# For demonstration, access the first record set
if len(record_set_ids) > 0:
    demo_rsid = record_set_ids[0]
    print(f"Preview of '{demo_rsid}':")
    display(dataframes[demo_rsid].head())

## 4. Exploratory Data Analysis (EDA)

We'll perform some fundamental EDA on the primary (first) record set loaded, referencing fields by their Croissant `@id` columns as shown above.

This includes basic summary, handling a numeric variable, filtering, normalization, and simple grouping by a categorical field.

In [ ]:
# Pick one record set for EDA
record_set_id = record_set_ids[0]  # Use the first (main) record set
df = dataframes[record_set_id]

print(f"Shape: {df.shape}")
print("Columns (@id):")
for i, c in enumerate(df.columns):
    print(f"  {i}: {c}")

# Attempt to auto-detect a numeric field by pandas dtype
numeric_cols = df.select_dtypes('number').columns.tolist()
if numeric_cols:
    numeric_field_id = numeric_cols[0]
else:
    print("No numeric field detected, picking a field by name like 'age' if exists.")
    possible_age = [col for col in df.columns if 'age' in col.lower()]
    numeric_field_id = possible_age[0] if possible_age else df.columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Filtering: e.g., show values over threshold (here: mean for example)
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].mean() if df.shape[0] > 0 else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")

    # Normalized column
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std(ddof=0) if filtered_df[numeric_field_id].std(ddof=0) > 0 else 1)
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Pick a possible group-by categorical field as next column
    possible_cats = df.select_dtypes('object').columns.tolist()
    group_field_id = possible_cats[0] if possible_cats else None

    if group_field_id:
        print(f"\nGrouping by field: {group_field_id}")
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
        print(grouped.head())
else:
    print("No well-typed numeric field available in the DataFrame.")

## 5. Visualization

We'll plot the distribution of the selected numeric variable, and (if a categorical grouping field is available) a boxplot by group. This uses matplotlib and seaborn for visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id], kde=True, bins=15, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by categorical field if possible
if 'group_field_id' in locals() and group_field_id:
    if df[group_field_id].nunique() < 20:  # Avoid too many categories
        plt.figure(figsize=(9,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
    else:
        print(f"Too many unique values in {group_field_id} for boxplot.")

## 6. Conclusion

In this notebook, we loaded and investigated a clinical dataset on second primary colorectal cancer survivors with tabular biomedical features as described by the Croissant schema. Data was loaded directly using each entity's `@id` via `mlcroissant` for reproducible, standards-based research workflows.

- Dataset: Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution
- Schema: [Croissant JSON-LD](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- Record sets, fields, and columns were referenced by `@id` throughout for clarity and robustness.

You can now proceed to domain-specific modelling, hypothesis testing, or further advanced analyses with this curated DataFrame structure.